# ⚡ Live Monitor – Quant Trader
**Workflow**: Connect to QTrader API → Display live engine status → PnL dashboard → Auto-refresh

> **Prerequisite**: QTrader must be running with `make docker-up` or `uvicorn qtrader.api.api:app`

---

In [ ]:
from qtrader.analyst import AnalystSession, RoleContext
import polars as pl
import time
import IPython.display as ipd

session = AnalystSession(role=RoleContext.TRADER)
session.info()

# Configure API endpoint
API_HOST = 'localhost'
API_PORT = 8000

## 1. Health Check

In [ ]:
is_alive = session.ping_live_api(host=API_HOST, port=API_PORT)
print(f"{'✅ QTrader API is ALIVE' if is_alive else '❌ QTrader API is UNREACHABLE – check bot status'}")

## 2. Live Status

In [ ]:
if is_alive:
    status = session.connect_live_api(host=API_HOST, port=API_PORT)

    # Pretty-print
    import json
    print(json.dumps(status, indent=2, default=str))

    # Display as table
    flat = {k: str(v) for k, v in status.items() if not isinstance(v, dict)}
    status_df = pl.DataFrame({'Field': list(flat.keys()), 'Value': list(flat.values())})
    display(status_df)
else:
    print("⚠️  Running in demo mode with synthetic data since API is unreachable.")
    status = {'engine_status': 'DEMO', 'regime': 'bull', 'active_model': 'momentum_v1',
               'uptime_seconds': 3600, 'iteration': 1000, 'total_exposure_btc': 0.42}

## 3. Live Equity Simulation (auto-refresh every N seconds)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output

# Simulate equity curve from live data or synthetic fallback
df = session.sample_ohlcv(symbol='BTC', days=90)
df = session.make_returns(df)
df = df.with_columns(pl.when(pl.col('returns') > 0).then(1).otherwise(-1).alias('signal'))
bt = session.run_vector_backtest(df, signal_col='signal')
equity = bt['equity_curve'].to_numpy()

REFRESH_SECONDS = 0  # Set > 0 for auto-refresh loop
N_REFRESH = 1        # Increase for live monitoring loop

for _ in range(N_REFRESH):
    clear_output(wait=True)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor='#0f1117')
    for ax in axes:
        ax.set_facecolor('#0f1117')
        for sp in ax.spines.values(): sp.set_color('#334155')
        ax.tick_params(colors='#94a3b8')

    # Equity
    axes[0].plot(equity, color='#34d399', linewidth=1.5)
    axes[0].set_title('Equity Curve', color='#e2e8f0')

    # Drawdown
    roll_max = np.maximum.accumulate(equity)
    dd = (equity - roll_max) / roll_max
    axes[1].fill_between(range(len(dd)), dd, color='#f87171', alpha=0.7)
    axes[1].set_title('Drawdown', color='#e2e8f0')

    # Status gauge
    axes[2].text(0.5, 0.6, status.get('engine_status', '?'), ha='center', va='center',
                  fontsize=22, color='#34d399', transform=axes[2].transAxes, fontweight='bold')
    axes[2].text(0.5, 0.35, f"Regime: {status.get('regime', '?')}", ha='center',
                  fontsize=12, color='#a78bfa', transform=axes[2].transAxes)
    axes[2].text(0.5, 0.2, f"Model: {status.get('active_model', '?')}", ha='center',
                  fontsize=10, color='#94a3b8', transform=axes[2].transAxes)
    axes[2].set_title('Engine Status', color='#e2e8f0')
    axes[2].axis('off')

    plt.suptitle('QTrader Live Dashboard', color='#7dd3fc', fontsize=14)
    plt.tight_layout()
    plt.show()

    if REFRESH_SECONDS > 0:
        # Fetch fresh status
        try: status = session.connect_live_api(host=API_HOST, port=API_PORT)
        except: pass
        time.sleep(REFRESH_SECONDS)